In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df=pd.read_csv('diabetes_patients_10000_full.csv')

In [5]:
df.head()

,Patient_ID,Age,Gender,Blood_Sugar_Level,Creatinine,BMI,Urea,Cholesterol,HbA1c,Class
0,1,60.0,Male,96.0,1.34,22.5,23.0,176.0,8.6,Diabetic
1,2,77.0,Male,241.0,1.02,19.0,28.0,209.0,7.5,Diabetic
2,3,21.0,Male,229.0,1.02,25.9,32.0,151.0,9.1,Diabetic
3,4,30.0,Female,177.0,0.88,22.1,36.0,176.0,5.1,Diabetic
4,5,26.0,Female,178.0,1.20,31.8,44.0,287.0,5.2,Diabetic


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Patient_ID         10000 non-null  int64  
 1   Age                10000 non-null  float64
 2   Gender             10000 non-null  str    
 3   Blood_Sugar_Level  10000 non-null  float64
 4   Creatinine         10000 non-null  float64
 5   BMI                10000 non-null  float64
 6   Urea               10000 non-null  float64
 7   Cholesterol        10000 non-null  float64
 8   HbA1c              10000 non-null  float64
 9   Class              10000 non-null  str    
dtypes: float64(7), int64(1), str(2)
memory usage: 781.4 KB


In [6]:
df.isnull().sum()

Patient_ID            0
Age                  14
Gender               40
Blood_Sugar_Level    53
Creatinine           56
BMI                  51
Urea                 51
Cholesterol          32
HbA1c                 0
Class                 0
dtype: int64

In [7]:

from sklearn.impute import SimpleImputer

# Numerical columns
num_cols = df.select_dtypes(include="number").columns.drop(
    "Patient_ID", errors="ignore"
)

# Categorical columns
cat_cols = df.select_dtypes(include=["object", "string"]).columns.drop(
    "Class", errors="ignore"
)

# Fill numerical nulls with Median
num_imputer = SimpleImputer(strategy="median")
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Fill categorical nulls with Most Frequent
cat_imputer = SimpleImputer(strategy="most_frequent")
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])


In [8]:
df.isnull().sum()

Patient_ID           0
Age                  0
Gender               0
Blood_Sugar_Level    0
Creatinine           0
BMI                  0
Urea                 0
Cholesterol          0
HbA1c                0
Class                0
dtype: int64


encoding

In [12]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["Gender"] = le.fit_transform(df["Gender"])


split

In [13]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=["Class", "Patient_ID"], errors="ignore")
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logistic

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)
lr_model = make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000, random_state=42))
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\n Report:\n",classification_report(y_test, y_pred_lr))



Accuracy: 0.972

 Report:
               precision    recall  f1-score   support

    Diabetic       0.98      0.99      0.98      1829
Non-Diabetic       0.89      0.77      0.82       171

    accuracy                           0.97      2000
   macro avg       0.93      0.88      0.90      2000
weighted avg       0.97      0.97      0.97      2000



knn

In [17]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)
knn_model = make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=5))
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print("\nReport:\n",classification_report(y_test, y_pred_knn))


Accuracy: 0.964

Report:
               precision    recall  f1-score   support

    Diabetic       0.97      0.99      0.98      1829
Non-Diabetic       0.84      0.71      0.77       171

    accuracy                           0.96      2000
   macro avg       0.91      0.85      0.88      2000
weighted avg       0.96      0.96      0.96      2000



naive bais

In [18]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nReport:\n",classification_report(y_test, y_pred_nb))

Accuracy: 0.988

Report:
               precision    recall  f1-score   support

    Diabetic       0.99      1.00      0.99      1829
Non-Diabetic       1.00      0.86      0.92       171

    accuracy                           0.99      2000
   macro avg       0.99      0.93      0.96      2000
weighted avg       0.99      0.99      0.99      2000



SVC

In [24]:

from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC())
])

svm_params = {
    "model__kernel": ["rbf", "linear", 'poly'],
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", 0.01, 0.1, 1]
}
svm_grid = GridSearchCV(svm_pipe, svm_params,cv=5, scoring="f1_weighted",n_jobs=-1, verbose=1)
svm_grid.fit(X_train, y_train)
print("Best Parameters:", svm_grid.best_params_)
print("Best CV Score:", svm_grid.best_score_)
best_svm = svm_grid.best_estimator_
y_pred_svm = best_svm.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Parameters: {'model__C': 1, 'model__gamma': 1, 'model__kernel': 'poly'}
Best CV Score: 0.9888958482816589
Test Accuracy: 0.987
              precision    recall  f1-score   support

    Diabetic       0.99      0.99      0.99      1829
Non-Diabetic       0.91      0.94      0.92       171

    accuracy                           0.99      2000
   macro avg       0.95      0.96      0.96      2000
weighted avg       0.99      0.99      0.99      2000



Decision tree

In [32]:

from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
dt_model = DecisionTreeClassifier(random_state=42)
dt_params = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 3, 5, 7, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": [None, "balanced"]
}
dt_grid = GridSearchCV(
    dt_model, dt_params,
    cv=5, scoring="f1_weighted",
    n_jobs=-1, verbose=1
)
dt_grid.fit(X_train, y_train)
best_dt = dt_grid.best_estimator_
y_pred_dt = best_dt.predict(X_test)
print(classification_report(y_test, y_pred_dt))
print("Best Parameters:", dt_grid.best_params_)
print("Best CV Score:", dt_grid.best_score_)
print("Test Accuracy:", accuracy_score(y_test, y_pred_dt))


Fitting 5 folds for each of 216 candidates, totalling 1080 fits
              precision    recall  f1-score   support

    Diabetic       1.00      1.00      1.00      1829
Non-Diabetic       1.00      1.00      1.00       171

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

Best Parameters: {'class_weight': None, 'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV Score: 0.9996239518134983
Test Accuracy: 1.0


In [27]:
from sklearn.metrics import accuracy_score
results = pd.DataFrame({
    "Model": ["Logistic Regression","KNN","Naive Bayes","SVM (GridSearch)","Decision Tree (GridSearch)"],
    "Accuracy (%)": [
        accuracy_score(y_test, lr_model.predict(X_test)) * 100,
        accuracy_score(y_test, knn_model.predict(X_test)) * 100,
        accuracy_score(y_test, nb_model.predict(X_test)) * 100,
        accuracy_score(y_test, svm_grid.best_estimator_.predict(X_test)) * 100,
        accuracy_score(y_test, dt_grid.best_estimator_.predict(X_test)) * 100
    ]
})
results["Accuracy (%)"] = results["Accuracy (%)"].round(2)
results = results.sort_values(
    by="Accuracy (%)", ascending=False
).reset_index(drop=True)
print(results)


                        Model  Accuracy (%)
0  Decision Tree (GridSearch)         100.0
1                 Naive Bayes          98.8
2            SVM (GridSearch)          98.7
3         Logistic Regression          97.2
4                         KNN          96.4


In [ ]:
# =========================
# Apply All Classification Models
# =========================
# This section applies the complete model set from Classification Models.ipynb
# to the prepared X_train, X_test, y_train, y_test variables.

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)
from sklearn.base import clone

classification_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVC": SVC(probability=True, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "LightGBM": LGBMClassifier(random_state=42)
}

classification_results = []
classification_fitted = {}

for name, model in classification_models.items():
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    model.fit(X_train, y_train)
    classification_fitted[name] = model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan,
        "PR AUC": average_precision_score(y_test, y_prob) if y_prob is not None else np.nan
    })

    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

classification_results_df = pd.DataFrame(classification_results).sort_values(
    by="F1 Score", ascending=False
).reset_index(drop=True)
classification_results_df.round(4)


In [ ]:
# =========================
# Voting Classifier
# =========================
voting_model = VotingClassifier(
    estimators=[
        ("lr", clone(classification_models["Logistic Regression"])),
        ("rf", clone(classification_models["Random Forest"])),
        ("nb", clone(classification_models["Naive Bayes"])),
        ("xgb", clone(classification_models["XGBoost"]))
    ],
    voting="soft"
)
voting_model.fit(X_train, y_train)
y_pred = voting_model.predict(X_test)
y_prob = voting_model.predict_proba(X_test)[:, 1]

voting_result = {
    "Model": "Voting Classifier",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred, zero_division=0),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "PR AUC": average_precision_score(y_test, y_prob)
}
classification_results_df = pd.concat([classification_results_df, pd.DataFrame([voting_result])], ignore_index=True)
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

classification_results_df.sort_values("F1 Score", ascending=False).reset_index(drop=True).round(4)


In [ ]:
# =========================
# Stacking Classifier
# =========================
stacking_model = StackingClassifier(
    estimators=[
        ("lr", clone(classification_models["Logistic Regression"])),
        ("rf", clone(classification_models["Random Forest"])),
        ("nb", clone(classification_models["Naive Bayes"])),
        ("lgbm", clone(classification_models["LightGBM"]))
    ],
    final_estimator=XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss"),
    stack_method="predict_proba",
    n_jobs=-1
)
stacking_model.fit(X_train, y_train)
y_pred = stacking_model.predict(X_test)
y_prob = stacking_model.predict_proba(X_test)[:, 1]

stacking_result = {
    "Model": "Stacking Classifier",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred, zero_division=0),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "PR AUC": average_precision_score(y_test, y_prob)
}
classification_results_df = pd.concat([classification_results_df, pd.DataFrame([stacking_result])], ignore_index=True)
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

classification_results_df = classification_results_df.sort_values(
    "F1 Score", ascending=False
).reset_index(drop=True)
classification_results_df.round(4)
